<div style="padding:24px;border-radius:18px;background:linear-gradient(135deg,#081c33,#173b5e);color:#f7fbff">
  <div style="font-size:13px;letter-spacing:.12em;text-transform:uppercase;opacity:.8">Laboratorio de sistemas complejos · Python + Google Colab</div>
  <h1 style="margin:.35em 0 .2em">Align to alike, avoid aliens</h1>
  <h2 style="margin:.1em 0;font-weight:400">BOIDS multigrupo: reglas locales, segregación y fronteras emergentes</h2>
  <p style="max-width:850px;margin-top:16px">Tres bandadas, cardúmenes, enjambres o poblaciones de agentes. Nadie dirige al conjunto: cada agente solo observa una vecindad local. Aun así aparecen orden colectivo, agrupamiento, separación y memoria espacial.</p>
  <p style="margin-bottom:0;opacity:.8">Material elaborado por el profesor Sergio Gevatschnaider</p>
</div>

> **Idea central.** Este notebook no busca que “mires puntos moverse”, sino que puedas formular hipótesis, medir resultados y construir un diagrama de fases reproducible.

## Objetivos de aprendizaje

Al finalizar podrás:

1. Explicar cómo **separación**, **alineamiento** y **cohesión** producen flocking sin líder.
2. Extender BOIDS a tres grupos mediante **homofilia local** y **evitación intergrupal**.
3. Distinguir orden dinámico, segregación espacial y exposición a fronteras.
4. Construir un **diagrama de fases** mediante barridos de parámetros y réplicas aleatorias.
5. Analizar sensibilidad a condiciones iniciales, transitorios, persistencia e histéresis.
6. Discutir qué se puede —y qué no se puede— inferir sobre aves, peces, drones, redes sociales, naciones o agentes de IA.

### Recorrido

| Bloque | Pregunta | Producto |
|---|---|---|
| 1. Modelo | ¿Qué información local usa cada agente? | Ecuaciones y algoritmo |
| 2. Simulación | ¿Qué patrones aparecen con tres grupos? | Trayectorias y animación |
| 3. Medición | ¿Orden y segregación son lo mismo? | Series temporales y métricas |
| 4. Experimentos | ¿Qué regla causa qué patrón? | Comparación controlada |
| 5. Diagrama de fases | ¿Dónde cambia el régimen colectivo? | Mapas paramétricos |
| 6. Persistencia | ¿Las fronteras sobreviven al cambio de reglas? | Experimento de intervención |

## 1. De reglas locales a propiedades globales

El modelo BOIDS clásico de Craig Reynolds representa a cada agente $i$ mediante posición $\mathbf{x}_i(t)\in[0,L)^2$ y velocidad $\mathbf{v}_i(t)$. En cada paso, el agente responde a vecinos dentro de radios de percepción; no conoce el estado global.

Para una vecindad $\mathcal N_i$:

$$
\mathbf a_i=
w_s\mathbf S_i +
w_a\mathbf A_i +
w_c\mathbf C_i +
w_o\mathbf O_i +
\sigma\boldsymbol\eta_i.
$$

- **Separación $\mathbf S_i$:** alejarse de cualquier agente demasiado próximo; evita colisiones.
- **Alineamiento $\mathbf A_i$:** aproximar la velocidad a la de vecinos del mismo grupo.
- **Cohesión $\mathbf C_i$:** dirigirse hacia el centro local de vecinos semejantes.
- **Evitación intergrupal $\mathbf O_i$:** alejarse de vecinos de otros grupos.
- **Ruido $\boldsymbol\eta_i$:** perturbaciones, errores de percepción o decisiones no modeladas.

Actualizamos con Euler discreto y limitamos la rapidez:

$$
\mathbf v_i(t+1)=\operatorname{clip}_{v_{\max}}
\left[\mathbf v_i(t)+\Delta t\,\mathbf a_i(t)\right],\qquad
\mathbf x_i(t+1)=\big(\mathbf x_i(t)+\Delta t\,\mathbf v_i(t+1)\big)\bmod L.
$$

El módulo implementa un **toro**: salir por la derecha equivale a entrar por la izquierda. Así evitamos que una pared artificial sea confundida con una frontera social.

### “Align to alike, avoid aliens”: traducción operacional

En el código evitaremos usar *aliens* como etiqueta para personas. Formalmente hay solo agentes con una variable categórica $g_i\in\{0,1,2\}$:

$$
\mathcal N_i^{\text{same}}=\{j:d_{ij}<r_p,\ g_j=g_i\},\qquad
\mathcal N_i^{\text{other}}=\{j:d_{ij}<r_o,\ g_j\neq g_i\}.
$$

La frase se convierte en dos parámetros observables:

- $h$ (**homofilia dinámica**): intensidad de alineamiento/cohesión con el mismo grupo.
- $q$ (**evitación intergrupal**): intensidad de repulsión frente a otros grupos.

Esto es un **modelo generativo mínimo**, no una teoría de la conducta humana. Que una regla produzca segregación es una demostración de suficiencia computacional, no prueba de que esa regla sea la causa de una segregación real.

In [ ]:
# @title 2. Preparación del entorno (ejecutar primero)
import math, time, warnings
from dataclasses import dataclass, replace
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.colors import ListedColormap
import seaborn as sns

from IPython.display import HTML, display, clear_output

warnings.filterwarnings("ignore", category=RuntimeWarning)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 115
plt.rcParams["animation.html"] = "jshtml"

PALETTE = np.array(["#4C78A8", "#F58518", "#54A24B"])
GROUP_NAMES = np.array(["Grupo A", "Grupo B", "Grupo C"])
SEED = 42

print("Entorno listo · NumPy", np.__version__, "· Pandas", pd.__version__)

## 2. Implementación vectorizada

La matriz $\Delta_{ij}=\mathbf x_j-\mathbf x_i$ contiene todos los desplazamientos. En un toro usamos la convención de **imagen mínima**:

$$
\Delta_{ij}\leftarrow \Delta_{ij}-L\,\operatorname{round}(\Delta_{ij}/L).
$$

Así, dos agentes situados en $x=0.2$ y $x=19.8$ dentro de una caja $L=20$ están a distancia $0.4$, no $19.6$. La vectorización hace el código claro y rápido para fines docentes; su costo es $O(N^2)$ por paso. Para miles de agentes conviene usar árboles espaciales o *cell lists*.

In [ ]:
@dataclass(frozen=True)
class BoidsConfig:
    n_agents: int = 75
    n_groups: int = 3
    box_size: float = 20.0
    dt: float = 0.12
    perception_radius: float = 3.0
    separation_radius: float = 0.75
    outgroup_radius: float = 2.2
    separation: float = 1.35
    alignment: float = 0.85
    cohesion: float = 0.055
    outgroup_avoidance: float = 0.55
    noise: float = 0.035
    min_speed: float = 0.35
    max_speed: float = 1.35


class MultiGroupBoids:
    '''BOIDS bidimensional con tres grupos y condiciones periódicas.'''

    def __init__(self, config=BoidsConfig(), seed=42, init="mixed"):
        self.cfg = config
        self.rng = np.random.default_rng(seed)
        n, L, k = config.n_agents, config.box_size, config.n_groups
        self.groups = np.arange(n) % k
        self.rng.shuffle(self.groups)

        if init == "mixed":
            self.pos = self.rng.uniform(0, L, size=(n, 2))
        elif init == "segregated":
            centers = np.c_[
                L/2 + 0.28*L*np.cos(2*np.pi*np.arange(k)/k),
                L/2 + 0.28*L*np.sin(2*np.pi*np.arange(k)/k)
            ]
            self.pos = (centers[self.groups] + self.rng.normal(0, 0.9, (n, 2))) % L
        else:
            raise ValueError("init debe ser 'mixed' o 'segregated'")

        angles = self.rng.uniform(0, 2*np.pi, n)
        speeds = self.rng.uniform(config.min_speed, config.max_speed, n)
        self.vel = np.c_[np.cos(angles), np.sin(angles)] * speeds[:, None]
        self.t = 0

    def pair_geometry(self):
        L = self.cfg.box_size
        delta = self.pos[None, :, :] - self.pos[:, None, :]  # x_j - x_i
        delta -= L * np.round(delta / L)
        dist = np.linalg.norm(delta, axis=2)
        np.fill_diagonal(dist, np.inf)
        return delta, dist

    @staticmethod
    def _masked_mean(values, mask):
        count = mask.sum(axis=1, keepdims=True)
        total = (values * mask[..., None]).sum(axis=1)
        return np.divide(total, count, out=np.zeros_like(total), where=count > 0)

    def acceleration(self):
        c = self.cfg
        delta, dist = self.pair_geometry()
        same = self.groups[:, None] == self.groups[None, :]
        other = ~same
        np.fill_diagonal(other, False)

        # 1) Separación universal: ponderación inversa al cuadrado.
        close = dist < c.separation_radius
        safe_dist = np.maximum(dist, 1e-9)
        sep = (-delta / safe_dist[..., None]**2 * close[..., None]).sum(axis=1)

        # 2) Alineamiento con semejantes.
        same_near = same & (dist < c.perception_radius)
        mean_vel = self._masked_mean(
            np.broadcast_to(self.vel[None, :, :], delta.shape), same_near
        )
        has_same = same_near.any(axis=1)
        align = np.where(has_same[:, None], mean_vel - self.vel, 0.0)

        # 3) Cohesión: promedio de desplazamientos toroidales hacia semejantes.
        cohesion = self._masked_mean(delta, same_near)

        # 4) Evitación de otros grupos dentro de un radio específico.
        out_near = other & (dist < c.outgroup_radius)
        avoid = (-delta / safe_dist[..., None]**2 * out_near[..., None]).sum(axis=1)

        noise = self.rng.normal(0, 1, self.vel.shape)
        return (c.separation * sep + c.alignment * align +
                c.cohesion * cohesion + c.outgroup_avoidance * avoid +
                c.noise * noise)

    def step(self, steps=1):
        for _ in range(steps):
            c = self.cfg
            self.vel += c.dt * self.acceleration()
            speed = np.linalg.norm(self.vel, axis=1, keepdims=True)
            direction = self.vel / np.maximum(speed, 1e-12)
            target = np.clip(speed, c.min_speed, c.max_speed)
            self.vel = direction * target
            self.pos = (self.pos + c.dt * self.vel) % c.box_size
            self.t += 1
        return self

    def copy_state(self):
        return self.pos.copy(), self.vel.copy(), self.groups.copy()


def sanity_checks():
    cfg = BoidsConfig(n_agents=12)
    sim = MultiGroupBoids(cfg, seed=1)
    for _ in range(20):
        sim.step()
    speed = np.linalg.norm(sim.vel, axis=1)
    assert np.isfinite(sim.pos).all() and np.isfinite(sim.vel).all()
    assert (sim.pos >= 0).all() and (sim.pos < cfg.box_size).all()
    assert speed.max() <= cfg.max_speed + 1e-9
    assert speed.min() >= cfg.min_speed - 1e-9
    return "Pruebas básicas superadas ✓"

print(sanity_checks())

## 3. Cómo medir lo que emerge

Una imagen puede sugerir un patrón, pero no demuestra que exista. Usaremos cuatro métricas:

1. **Orden direccional**
$$P=\left\|\frac1N\sum_i \frac{\mathbf v_i}{\|\mathbf v_i\|}\right\|\in[0,1].$$
$P\approx1$ significa movimiento colectivo alineado; $P\approx0$ indica direcciones compensadas.

2. **Fracción de vecinos semejantes** $f_s$: entre los pares próximos, proporción que pertenece al mismo grupo.

3. **Segregación corregida por composición**
$$S=\frac{f_s-f_0}{1-f_0},\qquad f_0=\sum_g p_g^2.$$
$S\approx0$ corresponde a mezcla aleatoria; $S\to1$, a vecindarios homogéneos. Puede ser negativa si hay más contacto intergrupal que el esperado.

4. **Exposición a frontera** $B$: proporción de agentes que tiene al menos un vecino de otro grupo dentro del radio de medición. Una población puede estar muy segregada y conservar una frontera extensa; por eso $S$ y $B$ no son redundantes.

In [ ]:
def metrics(sim, radius=None):
    radius = radius or sim.cfg.perception_radius
    _, dist = sim.pair_geometry()
    near = dist < radius
    same = sim.groups[:, None] == sim.groups[None, :]

    upper = np.triu(near, 1)
    total_pairs = upper.sum()
    same_pairs = (upper & same).sum()
    same_fraction = same_pairs / total_pairs if total_pairs else np.nan

    proportions = np.bincount(sim.groups, minlength=sim.cfg.n_groups) / len(sim.groups)
    baseline = np.sum(proportions**2)
    segregation = ((same_fraction - baseline) / (1 - baseline)
                   if total_pairs and baseline < 1 else np.nan)

    unit = sim.vel / np.maximum(np.linalg.norm(sim.vel, axis=1, keepdims=True), 1e-12)
    polarization = np.linalg.norm(unit.mean(axis=0))
    boundary_exposure = (near & ~same).any(axis=1).mean()
    mean_neighbors = near.sum(axis=1).mean()

    return dict(
        t=sim.t,
        polarization=polarization,
        segregation=segregation,
        same_neighbor_fraction=same_fraction,
        boundary_exposure=boundary_exposure,
        mean_neighbors=mean_neighbors,
    )


def run_simulation(config=BoidsConfig(), steps=400, seed=42, init="mixed",
                   sample_every=5, keep_states=False):
    sim = MultiGroupBoids(config, seed=seed, init=init)
    records, states = [], []
    for t in range(steps + 1):
        if t % sample_every == 0:
            records.append(metrics(sim))
            if keep_states:
                states.append(sim.copy_state())
        if t < steps:
            sim.step()
    return sim, pd.DataFrame(records), states


def snapshot(sim, ax=None, title=None, arrows=True):
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 7))
    L = sim.cfg.box_size
    for g in range(sim.cfg.n_groups):
        m = sim.groups == g
        ax.scatter(sim.pos[m, 0], sim.pos[m, 1], s=42, color=PALETTE[g],
                   label=GROUP_NAMES[g], alpha=.88, edgecolor="white", linewidth=.35)
        if arrows:
            ax.quiver(sim.pos[m, 0], sim.pos[m, 1], sim.vel[m, 0], sim.vel[m, 1],
                      color=PALETTE[g], angles="xy", scale_units="xy", scale=2.8,
                      width=.003, alpha=.65)
    ax.set(xlim=(0, L), ylim=(0, L), xlabel="x", ylabel="y", aspect="equal")
    ax.set_title(title or f"Estado en t={sim.t}")
    ax.legend(loc="upper right", frameon=True, ncols=1)
    return ax

## 4. Primera experiencia: de una mezcla a tres colectivos

**Hipótesis:** con alineamiento y cohesión intragrupal moderados, más evitación intergrupal, aumentará la segregación aun cuando el orden direccional global no necesariamente sea alto. Tres grupos pueden formar tres bandadas que viajan en direcciones distintas: orden local alto, orden global bajo.

In [ ]:
# @title Simulación base y diagnóstico visual
base_cfg = BoidsConfig(
    n_agents=75,
    alignment=0.90,
    cohesion=0.065,
    outgroup_avoidance=0.70,
    noise=0.03,
)
base_sim, base_hist, base_states = run_simulation(
    base_cfg, steps=500, seed=SEED, sample_every=5, keep_states=True
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.6), constrained_layout=True)

# Reconstrucción del estado inicial para compararlo con el final.
initial = MultiGroupBoids(base_cfg, seed=SEED, init="mixed")
snapshot(initial, axes[0], "Inicio: mezcla aleatoria", arrows=False)
snapshot(base_sim, axes[1], "Final: patrón emergente", arrows=True)
plt.show()

display(base_hist.tail(1).round(3).rename(index={base_hist.index[-1]: "valor final"}))

In [ ]:
# @title Evolución temporal: orden, segregación y frontera
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharex=True, constrained_layout=True)
series = [
    ("polarization", "Orden direccional P", "#4C78A8", (0, 1)),
    ("segregation", "Segregación S", "#F58518", (-.1, 1)),
    ("boundary_exposure", "Exposición a frontera B", "#54A24B", (0, 1)),
]
for ax, (col, label, color, ylim) in zip(axes, series):
    ax.plot(base_hist["t"], base_hist[col], lw=2.2, color=color)
    ax.set(xlabel="Paso", ylabel=label, ylim=ylim)
    ax.axhline(0, color="gray", lw=.7)
fig.suptitle("Una misma trayectoria, tres propiedades colectivas distintas", y=1.03)
plt.show()

In [ ]:
# @title Animación de la trayectoria (puede tardar unos segundos)
def animate_states(states, config, interval=55, stride=2):
    shown = states[::stride]
    fig, ax = plt.subplots(figsize=(6.4, 6.4))
    ax.set(xlim=(0, config.box_size), ylim=(0, config.box_size), aspect="equal",
           xlabel="x", ylabel="y")
    scatters = []
    groups = shown[0][2]
    for g in range(config.n_groups):
        sc = ax.scatter([], [], s=38, color=PALETTE[g], label=GROUP_NAMES[g],
                        edgecolor="white", linewidth=.3)
        scatters.append(sc)
    ax.legend(loc="upper right")
    title = ax.set_title("")

    def update(frame):
        pos, vel, grp = shown[frame]
        for g, sc in enumerate(scatters):
            sc.set_offsets(pos[grp == g])
        title.set_text(f"BOIDS multigrupo · muestra {frame+1}/{len(shown)}")
        return [*scatters, title]

    ani = animation.FuncAnimation(fig, update, frames=len(shown), interval=interval,
                                  blit=False, repeat=True)
    plt.close(fig)
    return ani

ani = animate_states(base_states, base_cfg, stride=2)
display(HTML(ani.to_jshtml()))

## 5. Laboratorio interactivo

Mueve los controles y pulsa **Ejecutar experimento**. No cambies todo a la vez si quieres hacer inferencia causal: modifica un parámetro, conserva la semilla y compara.

- **Alineamiento:** sincroniza velocidades del mismo grupo.
- **Cohesión:** compacta cada grupo.
- **Evitación intergrupal:** reduce contacto entre grupos.
- **Ruido:** debilita la coordinación y puede disolver fronteras.

In [ ]:
# @title Controles interactivos (ipywidgets)
try:
    import ipywidgets as widgets

    align_w = widgets.FloatSlider(value=.9, min=0, max=1.8, step=.1, description="Alineam.")
    cohesion_w = widgets.FloatSlider(value=.06, min=0, max=.15, step=.01, description="Cohesión")
    avoid_w = widgets.FloatSlider(value=.7, min=0, max=1.8, step=.1, description="Evitación")
    noise_w = widgets.FloatSlider(value=.03, min=0, max=.20, step=.01, description="Ruido")
    seed_w = widgets.IntSlider(value=42, min=0, max=100, step=1, description="Semilla")
    run_button = widgets.Button(description="Ejecutar experimento", button_style="primary")
    out = widgets.Output()

    def run_widget(_):
        with out:
            clear_output(wait=True)
            cfg = replace(base_cfg, alignment=align_w.value, cohesion=cohesion_w.value,
                          outgroup_avoidance=avoid_w.value, noise=noise_w.value)
            sim, hist, _ = run_simulation(cfg, steps=350, seed=seed_w.value, sample_every=5)
            fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
            snapshot(sim, axes[0], "Estado final", arrows=False)
            axes[1].plot(hist.t, hist.polarization, label="Orden P")
            axes[1].plot(hist.t, hist.segregation, label="Segregación S")
            axes[1].plot(hist.t, hist.boundary_exposure, label="Frontera B")
            axes[1].set(xlabel="Paso", ylabel="Métrica", ylim=(-.1, 1.05), title="Dinámica")
            axes[1].legend()
            plt.show()
            display(hist.tail(20).mean(numeric_only=True).to_frame("promedio final").T.round(3))

    run_button.on_click(run_widget)
    display(widgets.VBox([
        widgets.HTML("<b>Reglas locales</b>"),
        widgets.HBox([align_w, cohesion_w]),
        widgets.HBox([avoid_w, noise_w]),
        seed_w, run_button, out
    ]))
    run_widget(None)
except Exception as exc:
    print("Los widgets no están disponibles. Ejecuta manualmente la simulación base.", exc)

## 6. Experimentos controlados: cuatro regímenes candidatos

La etiqueta de un régimen es una interpretación resumida, no una verdad ontológica. Compararemos promedios de la cola temporal para evitar confundir el transitorio inicial con el comportamiento persistente.

In [ ]:
# @title Comparación de escenarios con la misma semilla
scenarios = {
    "Desorden ruidoso": dict(alignment=.15, cohesion=.01, outgroup_avoidance=.05, noise=.16),
    "Bandada mixta": dict(alignment=1.15, cohesion=.045, outgroup_avoidance=0.0, noise=.02),
    "Tres bandadas": dict(alignment=1.05, cohesion=.075, outgroup_avoidance=.85, noise=.025),
    "Fragmentación": dict(alignment=.40, cohesion=.018, outgroup_avoidance=1.55, noise=.08),
}

scenario_rows, scenario_sims = [], {}
for name, pars in scenarios.items():
    cfg = replace(base_cfg, **pars)
    sim, hist, _ = run_simulation(cfg, steps=450, seed=23, sample_every=5)
    tail = hist.tail(20).mean(numeric_only=True)
    scenario_rows.append({"escenario": name, **tail.to_dict()})
    scenario_sims[name] = sim

scenario_df = pd.DataFrame(scenario_rows).set_index("escenario")
display(scenario_df[["polarization", "segregation", "boundary_exposure",
                     "mean_neighbors"]].round(3))

fig, axes = plt.subplots(2, 2, figsize=(11, 10), constrained_layout=True)
for ax, (name, sim) in zip(axes.ravel(), scenario_sims.items()):
    snapshot(sim, ax, name, arrows=False)
    ax.get_legend().remove()
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="outside upper center", ncols=3)
plt.show()

### Lectura esperada

- **Bandada mixta:** puede tener $P$ alto con $S$ cercano a cero. Coordinación no implica segregación.
- **Tres bandadas:** puede tener $S$ alto y $P$ global intermedio o bajo si cada grupo toma una dirección diferente.
- **Fragmentación:** una evitación extrema reduce vecinos, de modo que el índice $S$ puede volverse inestable. Siempre hay que leerlo junto con densidad local y exposición.
- **Desorden ruidoso:** no toda ausencia de orden equivale a mezcla social; puede ser simplemente pérdida de coordinación cinemática.

## 7. Diagrama de fases

Un diagrama de fases resume el régimen macroscópico en función de parámetros microscópicos. Barreremos:

- eje horizontal: cohesión intragrupal $w_c$;
- eje vertical: evitación intergrupal $w_o$.

Para cada punto ejecutamos varias semillas y promediamos el tramo final. Esto reduce el riesgo de dibujar como “fase” un accidente de una sola inicialización.

> **Costo computacional.** El barrido rápido usa pocos agentes y una grilla 7×7. Aumenta `repeats`, `steps` y la resolución solo después de verificar que la versión corta funciona.

In [ ]:
def phase_sweep(cohesion_values, avoidance_values, base=None,
                steps=260, repeats=2, seed=1000):
    base = base or replace(base_cfg, n_agents=54, alignment=.95, noise=.035)
    rows = []
    total = len(cohesion_values) * len(avoidance_values) * repeats
    done = 0
    start = time.time()
    for avoid in avoidance_values:
        for coh in cohesion_values:
            for rep in range(repeats):
                cfg = replace(base, cohesion=float(coh), outgroup_avoidance=float(avoid))
                # La misma semilla por réplica se reutiliza en toda la grilla:
                # comparación pareada que reduce variación ajena a los parámetros.
                _, hist, _ = run_simulation(cfg, steps=steps, seed=seed+rep,
                                            sample_every=10)
                tail = hist.tail(max(5, len(hist)//4)).mean(numeric_only=True)
                rows.append({"cohesion": coh, "avoidance": avoid, "rep": rep,
                             "P": tail.polarization, "S": tail.segregation,
                             "B": tail.boundary_exposure,
                             "neighbors": tail.mean_neighbors})
                done += 1
    result = pd.DataFrame(rows)
    print(f"Barrido terminado: {total} simulaciones en {time.time()-start:.1f} s")
    return result


cohesion_grid = np.round(np.linspace(0.00, 0.12, 7), 3)
avoidance_grid = np.round(np.linspace(0.00, 1.50, 7), 3)
phase_raw = phase_sweep(cohesion_grid, avoidance_grid, steps=240, repeats=2)
phase = phase_raw.groupby(["avoidance", "cohesion"], as_index=False).mean(numeric_only=True)
phase.head()

In [ ]:
# @title Mapas cuantitativos del diagrama de fases
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), constrained_layout=True)
for ax, col, title, cmap, bounds in [
    (axes[0], "P", "Orden direccional P", "Blues", (0, 1)),
    (axes[1], "S", "Segregación S", "Oranges", (-.1, 1)),
    (axes[2], "B", "Exposición a frontera B", "Greens", (0, 1)),
]:
    table = phase.pivot(index="avoidance", columns="cohesion", values=col)
    sns.heatmap(table, ax=ax, cmap=cmap, vmin=bounds[0], vmax=bounds[1],
                annot=True, fmt=".2f", cbar_kws={"shrink": .75})
    ax.invert_yaxis()
    ax.set(title=title, xlabel="Cohesión intragrupal", ylabel="Evitación intergrupal")
plt.show()

In [ ]:
# @title Clasificación pedagógica de regímenes
def classify_regime(row):
    # Umbrales explícitos y editables: sirven para resumir, no para ocultar los datos.
    if row.neighbors < 2.0:
        return "Fragmentado"
    if row.S >= .38 and row.P >= .45:
        return "Bandadas segregadas"
    if row.S >= .38:
        return "Segregado sin rumbo común"
    if row.P >= .55:
        return "Bandada mixta"
    return "Mixto/desordenado"

phase["regime"] = phase.apply(classify_regime, axis=1)
regime_order = ["Mixto/desordenado", "Bandada mixta", "Segregado sin rumbo común",
                "Bandadas segregadas", "Fragmentado"]
phase["regime_code"] = pd.Categorical(phase.regime, categories=regime_order).codes
regime_table = phase.pivot(index="avoidance", columns="cohesion", values="regime_code")

colors = ["#B8B8B8", "#4C78A8", "#F2CF5B", "#F58518", "#8E5EA2"]
fig, ax = plt.subplots(figsize=(8.4, 6.2))
sns.heatmap(regime_table, cmap=ListedColormap(colors), vmin=-.5, vmax=4.5,
            linewidths=.6, linecolor="white", cbar=False, ax=ax)
ax.invert_yaxis()
ax.set(title="Diagrama de regímenes (clasificación explícita)",
       xlabel="Cohesión intragrupal", ylabel="Evitación intergrupal")
handles = [plt.Line2D([0], [0], marker="s", linestyle="", markersize=11,
                      markerfacecolor=c, markeredgecolor="none", label=lab)
           for c, lab in zip(colors, regime_order)]
ax.legend(handles=handles, bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
plt.show()

### Cómo interpretar el diagrama sin sobreafirmar

1. Las fronteras de color son **zonas de transición estimadas**, no discontinuidades matemáticas demostradas.
2. Los umbrales de clasificación son visibles en `classify_regime`; cambia los umbrales y comprueba robustez.
3. Una fase exige estabilidad frente a tamaño poblacional, duración, semillas y resolución de la grilla.
4. Si la desviación entre réplicas es grande, el dato correcto es “resultado incierto o multestable”, no una categoría forzada.

In [ ]:
# @title Incertidumbre entre réplicas
uncertainty = (phase_raw.groupby(["avoidance", "cohesion"])
               .agg(S_mean=("S", "mean"), S_sd=("S", "std"),
                    P_mean=("P", "mean"), P_sd=("P", "std"))
               .reset_index())
display(uncertainty.sort_values("S_sd", ascending=False).head(10).round(3))

## 8. Persistencia de fronteras: una intervención

“Persistente” no significa eterna. Lo probaremos con un diseño en dos etapas:

1. **Formación:** alta evitación intergrupal durante 350 pasos.
2. **Intervención:** reducimos la evitación casi a cero sin reiniciar posiciones ni velocidades.

Si $S$ cae instantáneamente, la frontera era solo una respuesta contemporánea. Si decae lentamente, el sistema exhibe **memoria estructural**: la configuración producida en el pasado condiciona el presente.

In [ ]:
def intervention_experiment(seed=7, pre_steps=350, post_steps=450,
                            avoid_before=1.1, avoid_after=.03):
    cfg_before = replace(base_cfg, cohesion=.075, alignment=1.0,
                         outgroup_avoidance=avoid_before, noise=.025)
    sim = MultiGroupBoids(cfg_before, seed=seed, init="mixed")
    rows = []
    for _ in range(pre_steps):
        if sim.t % 5 == 0:
            rows.append({**metrics(sim), "stage": "formación"})
        sim.step()

    # Intervención: el estado se conserva; solo cambia la regla.
    sim.cfg = replace(sim.cfg, outgroup_avoidance=avoid_after)
    switch_t = sim.t
    for _ in range(post_steps + 1):
        if sim.t % 5 == 0:
            rows.append({**metrics(sim), "stage": "evitación reducida"})
        sim.step()
    return sim, pd.DataFrame(rows), switch_t


intervention_sim, intervention_hist, switch_t = intervention_experiment()

fig, ax = plt.subplots(figsize=(10, 4.6))
ax.plot(intervention_hist.t, intervention_hist.segregation, lw=2.3,
        color="#F58518", label="Segregación S")
ax.plot(intervention_hist.t, intervention_hist.boundary_exposure, lw=2.0,
        color="#54A24B", label="Exposición B")
ax.axvline(switch_t, color="#333333", ls="--", lw=1.5,
           label="Intervención: baja la evitación")
ax.set(xlabel="Paso", ylabel="Métrica", ylim=(-.1, 1.05),
       title="¿Sobrevive la frontera al cambio de regla?")
ax.legend(ncols=3, loc="upper center", bbox_to_anchor=(.5, 1.16))
plt.show()

## 9. Sensibilidad y reproducibilidad

Los sistemas no lineales pueden amplificar pequeñas diferencias. Por eso una semilla no es “ruido que hay que ocultar”: es una dimensión del experimento.

In [ ]:
# @title Distribución de resultados sobre múltiples semillas
def seed_ensemble(config, seeds=range(12), steps=400):
    rows = []
    for seed in seeds:
        _, hist, _ = run_simulation(config, steps=steps, seed=seed, sample_every=10)
        tail = hist.tail(10).mean(numeric_only=True)
        rows.append({"seed": seed, "P": tail.polarization, "S": tail.segregation,
                     "B": tail.boundary_exposure, "neighbors": tail.mean_neighbors})
    return pd.DataFrame(rows)

ensemble = seed_ensemble(base_cfg)
display(ensemble.describe().round(3))

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), constrained_layout=True)
for ax, col, color in zip(axes, ["P", "S", "B"], PALETTE):
    sns.stripplot(data=ensemble, y=col, ax=ax, color=color, size=7, jitter=.08)
    ax.axhline(ensemble[col].mean(), color="black", lw=1, ls="--")
    ax.set(title=f"{col} entre semillas", xlabel="", ylabel=col, ylim=(-.05, 1.05))
plt.show()

## 10. De aves a drones, redes y agentes de IA

| Dominio | Agente | Vecindad | “Alineamiento” | Riesgo de extrapolación |
|---|---|---|---|---|
| Aves o peces | individuo | distancia/sensores | dirección y velocidad | omitir aerodinámica, depredación y jerarquías |
| Drones | vehículo | radio, red o visión | rumbo/protocolo | ignorar latencia, obstáculos y seguridad |
| Red social | cuenta/persona | enlaces o exposición | opinión/acción | confundir homofilia con influencia |
| Naciones o ejércitos | organización | alianzas, geografía | postura estratégica | antropomorfizar y borrar instituciones |
| Agentes de IA | proceso/modelo | canal de comunicación | política o representación | suponer autonomía donde hay diseño humano |

### Principio metodológico

La transferencia válida no consiste en decir “las naciones son pájaros”, sino en preguntar si distintos sistemas comparten una **estructura de interacción local**, identificar qué variables corresponden y buscar datos que puedan refutar la analogía.

### Cuatro límites importantes

- El modelo asigna identidades fijas; en sistemas sociales reales las identidades pueden ser múltiples y cambiar.
- La geometría física no equivale automáticamente a distancia cultural, política o informacional.
- Los agentes humanos anticipan, interpretan reglas y modifican instituciones.
- Los resultados pueden alimentar narrativas normativas; aquí son resultados descriptivos de reglas explícitas.

## 11. Actividades propuestas

### Nivel 1 — Comprensión

1. Pon `outgroup_avoidance=0`. ¿Puede aparecer segregación solo por cohesión intragrupal?
2. Pon `cohesion=0` y conserva alineamiento. ¿Se forman bandadas compactas o corrientes dispersas?
3. Aumenta `noise` gradualmente. Estima el punto donde $P$ deja de ser estable.

### Nivel 2 — Diseño experimental

4. Repite el diagrama con 30, 60 y 120 agentes. ¿Las fronteras de fase cambian?
5. Compara inicialización `mixed` y `segregated` con los mismos parámetros. Si el estado final depende del inicio, documenta multestabilidad o histéresis.
6. Sustituye el radio métrico por los $k$ vecinos más próximos. ¿Qué propiedades dependen de densidad?

### Nivel 3 — Extensión

7. Añade obstáculos o recursos y mide si las fronteras coinciden con el ambiente.
8. Permite alineamiento intergrupal positivo. ¿Cuándo surge coordinación global sin mezcla espacial?
9. Haz que la identidad del grupo pueda cambiar según vecinos. ¿Aparecen consenso, ciclos o dominios móviles?
10. Para drones, añade retardo de comunicación y perturbaciones; evalúa colisiones y robustez.

### Entrega sugerida

Formula una hipótesis, define variable independiente, métricas y controles; ejecuta al menos 10 semillas; informa promedio, dispersión y un caso atípico; concluye sin extrapolar más allá del modelo.

## 12. Síntesis

1. **Reglas simples no implican resultados simples.** Las interacciones locales se realimentan y crean patrones macroscópicos.
2. **Orden no es segregación.** $P$, $S$ y $B$ responden preguntas diferentes.
3. **Una imagen no es evidencia suficiente.** Se necesitan métricas, réplicas, controles y análisis de sensibilidad.
4. **Las fronteras pueden tener memoria.** Un patrón espacial puede persistir aun después de cambiar la regla que lo formó.
5. **Una analogía es una hipótesis, no una identidad.** El modelo puede inspirar preguntas sobre peces, drones, redes o IA, pero no reemplaza la teoría ni los datos de cada dominio.

### Referencias conceptuales

- Reynolds, C. W. (1987). *Flocks, Herds and Schools: A Distributed Behavioral Model*.
- Vicsek, T. et al. (1995). *Novel Type of Phase Transition in a System of Self-Driven Particles*.
- Schelling, T. C. (1971). *Dynamic Models of Segregation*.
- Wolfram, C. *Phase diagram of boids: complex global behavior emerging from simple local rules*, Wolfram Community.

> El modelo de este notebook es una implementación didáctica original en Python/NumPy inspirada en la familia BOIDS; no es una traducción línea por línea del notebook de Wolfram Language.

## Apéndice A — Exportar resultados

In [ ]:
# @title Guardar tablas para análisis posterior
# Descomenta si quieres descargar los resultados desde Colab.
# phase_raw.to_csv("boids_phase_diagram_raw.csv", index=False)
# ensemble.to_csv("boids_seed_ensemble.csv", index=False)

print("Objetos disponibles:")
print("  base_hist            → trayectoria base")
print("  scenario_df          → comparación de escenarios")
print("  phase_raw / phase    → diagrama de fases")
print("  intervention_hist    → persistencia de fronteras")
print("  ensemble             → sensibilidad a semillas")